In [2]:
import pandas as pd


In [3]:
df=pd.read_csv('../data/final_dataset.csv')

In [4]:
df.head()

,text,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 577135 entries, 0 to 577134
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   text       577135 non-null  object
 1   sentiment  577135 non-null  object
dtypes: object(2)
memory usage: 8.8+ MB


In [6]:
df.shape

(577135, 2)

In [7]:
df.isnull().sum()

text         0
sentiment    0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(161746)

In [9]:
df = df.drop_duplicates()

In [10]:
label_map = {
    'negative': 0,
    'neutral': 1,
    'positive': 2
}

df['sentiment'] = df['sentiment'].map(label_map)

In [11]:
from sklearn.model_selection import train_test_split
X = df['text']
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [13]:
X_train_tfidf.shape

(290772, 5000)

In [14]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42)
model.fit(X_train_tfidf, y_train)
y_pred_lr = model.predict(X_test_tfidf)


In [15]:
from sklearn.naive_bayes import MultinomialNB
model_nb = MultinomialNB()

model_nb.fit(X_train_tfidf, y_train)

y_pred_nb = model_nb.predict(X_test_tfidf)

In [16]:
from sklearn.svm import LinearSVC
model_svm = LinearSVC()

model_svm.fit(X_train_tfidf, y_train)

y_pred_svm = model_svm.predict(X_test_tfidf)

In [17]:
from sklearn.metrics import accuracy_score
acc_lr=accuracy_score(y_test, y_pred_lr)
print(f"Logistic Regression Accuracy: {acc_lr:.4f}")


Logistic Regression Accuracy: 0.8866


In [18]:
acc_nb=accuracy_score(y_test, y_pred_nb)
print(f"MultinomialNB Accuracy: {acc_nb:.4f}")

MultinomialNB Accuracy: 0.7971


In [19]:
acc_svm=accuracy_score(y_test, y_pred_svm)
print(f"Linear SVM Accuracy: {acc_svm:.4f}")

Linear SVM Accuracy: 0.8870


In [20]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

           0       0.80      0.79      0.80     22907
           1       0.53      0.25      0.34      8051
           2       0.92      0.96      0.94     93659

    accuracy                           0.89    124617
   macro avg       0.75      0.67      0.69    124617
weighted avg       0.87      0.89      0.88    124617



In [21]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_nb))

              precision    recall  f1-score   support

           0       0.63      0.52      0.57     22907
           1       0.54      0.07      0.12      8051
           2       0.83      0.93      0.88     93659

    accuracy                           0.80    124617
   macro avg       0.67      0.50      0.52    124617
weighted avg       0.77      0.80      0.77    124617



In [22]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_svm)) 

              precision    recall  f1-score   support

           0       0.79      0.80      0.80     22907
           1       0.57      0.19      0.29      8051
           2       0.92      0.97      0.94     93659

    accuracy                           0.89    124617
   macro avg       0.76      0.65      0.68    124617
weighted avg       0.87      0.89      0.87    124617



In [23]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['newton-cg'],
    'class_weight': [None, 'balanced']
}

lr = LogisticRegression(random_state=42, max_iter=1000)
grid_search = GridSearchCV(
    lr,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

In [24]:
grid_search.fit(X_train_tfidf, y_train)

best_model = grid_search.best_estimator_
print("Best params:", grid_search.best_params_)
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")

y_pred_best = best_model.predict(X_test_tfidf)
acc_best = accuracy_score(y_test, y_pred_best)
print(f"Test accuracy with tuned Logistic Regression: {acc_best:.4f}")

Fitting 3 folds for each of 16 candidates, totalling 48 fits


f:\Desktop\AI -based Sentiment Analysis with Chatbot\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
24 fits failed out of a total of 48.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
24 fits failed with the following error:
Traceback (most recent call last):
  File "f:\Desktop\AI -based Sentiment Analysis with Chatbot\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "f:\Desktop\AI -based Sentiment Analysis with Chatbot\.venv\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "f:\Deskto

Best params: {'C': 1, 'class_weight': None, 'penalty': 'l2', 'solver': 'newton-cg'}
Best CV accuracy: 0.8853
Test accuracy with tuned Logistic Regression: 0.8868


In [ ]:
import os
import joblib

os.makedirs('../app/models', exist_ok=True)
joblib.dump(best_model, '../app/models/logistic_regression_model.pkl')

['../app/models/logistic_regression_model.pkl']

In [27]:
joblib.dump(vectorizer, '../app/models/tfidf_vectorizer.pkl')

['../app/models/tfidf_vectorizer.pkl']